# 🎙️ Voice2Vision — One-Click GPU Web App

Convert spoken voice into **Images** and **Animated Videos** using authentic AI models — running **100% Free** on Google Colab's T4 GPU.

### Models Used:
1. **Speech-to-Text**: `OpenMOSS-Team/MOSS-Transcribe-Diarize` (0.9B CausalLM)
2. **Image Generation**: `dreamlike-art/dreamlike-diffusion-1.0` (Original Stable Diffusion, **no watermarks**)
3. **Video Generation**: `AnimateDiffPipeline` with `AnimateLCM` + `emilianJR/epiCRealism` (Real 16-frame AI Video)

---
**Important:** Ensure you selected **Runtime > Change runtime type > T4 GPU** before running.

## 🚀 One-Click Run (Everything in One Cell)
Click the **Play (▶)** button on the cell below. It will automatically install dependencies, load the AI models, and launch your Gradio Web App with a free public link!

In [ ]:
# ============================================================
# 1. GPU Check & Dependencies
# ============================================================
!nvidia-smi

import os, sys

# Clone and install MOSS if not already present
REPO_DIR = '/content/MOSS-Transcribe-Diarize'
if not os.path.exists(os.path.join(REPO_DIR, 'moss_transcribe_diarize')):
    print("\n[1/4] Cloning MOSS-Transcribe-Diarize repository...")
    !git clone https://github.com/OpenMOSS/MOSS-Transcribe-Diarize.git {REPO_DIR}
    !pip install -q -e {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("\n[2/4] Installing required AI packages (Diffusers, Accelerate, Gradio)...")
!pip install -q diffusers transformers accelerate peft "torchao>=0.16.0" gradio

# ============================================================
# 2. Load AI Models onto GPU
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from diffusers import StableDiffusionPipeline, AnimateDiffPipeline, LCMScheduler, MotionAdapter
from diffusers.utils import export_to_gif
from moss_transcribe_diarize import parse_transcript
from moss_transcribe_diarize.inference_utils import (
    build_transcription_messages,
    generate_transcription,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
moss_dtype = torch.bfloat16 if device.type == 'cuda' else torch.float32
pipe_dtype = torch.float16 if device.type == 'cuda' else torch.float32

print(f"\nRuntime Device: {device}")

# 1. Load MOSS Speech Model
print("\n[3/4] Loading MOSS-Transcribe-Diarize Speech Model...")
moss_model = AutoModelForCausalLM.from_pretrained(
    'OpenMOSS-Team/MOSS-Transcribe-Diarize',
    trust_remote_code=True,
    dtype='auto',
    attn_implementation='sdpa' if device.type == 'cuda' else None,
).to(dtype=moss_dtype, device=device).eval()

moss_processor = AutoProcessor.from_pretrained(
    'OpenMOSS-Team/MOSS-Transcribe-Diarize',
    trust_remote_code=True,
)

# 2. Load Dreamlike Diffusion (Images)
print("Loading Dreamlike Diffusion 1.0 (No watermark)...")
image_pipe = StableDiffusionPipeline.from_pretrained(
    'dreamlike-art/dreamlike-diffusion-1.0',
    torch_dtype=pipe_dtype,
    use_safetensors=True,
).to(device)

# 3. Load AnimateDiff + AnimateLCM (Videos)
print("Loading AnimateDiff + AnimateLCM (Real AI Video)...")
adapter = MotionAdapter.from_pretrained(
    'wangfuyun/AnimateLCM',
    torch_dtype=pipe_dtype,
)
video_pipe = AnimateDiffPipeline.from_pretrained(
    'emilianJR/epiCRealism',
    motion_adapter=adapter,
    torch_dtype=pipe_dtype,
)
video_pipe.scheduler = LCMScheduler.from_config(
    video_pipe.scheduler.config,
    beta_schedule='linear',
)
video_pipe.load_lora_weights(
    'wangfuyun/AnimateLCM',
    weight_name='AnimateLCM_sd15_t2v_lora.safetensors',
    adapter_name='lcm-lora',
)
video_pipe.set_adapters(['lcm-lora'], [0.8])
video_pipe.enable_model_cpu_offload()

print("\n✅ All models loaded into GPU!")

# ============================================================
# 3. Launch Gradio Web App
# ============================================================
import gradio as gr

NEGATIVE_PROMPT = (
    "bad pose, awkward pose, unnatural posture, broken posture, "
    "broken fingers, malformed fingers, extra fingers, missing fingers, "
    "deformed hands, deformed feet, deformed legs, deformed arms, "
    "long limbs, short limbs, twisted limbs, unnatural joints, "
    "facial asymmetry, distorted mouth, distorted teeth, distorted nose, "
    "distorted ears, deformed eyes, uneven eyes, blinking artifacts, "
    "face melting, face warping, face morphing, identity drift, "
    "skin flickering, hair flickering, hair changing, "
    "clothing deformation, clothing flickering, clothing morphing, "
    "accessory deformation, jewelry deformation"
)

def transcribe_voice(audio_file):
    if not audio_file:
        return "❌ Please record or upload an audio file."
    try:
        messages = build_transcription_messages(audio_file)
        result = generate_transcription(
            moss_model,
            moss_processor,
            messages,
            max_new_tokens=2048,
            do_sample=False,
            device=device,
            dtype=moss_dtype,
        )
        segments = parse_transcript(result["text"])
        transcript = " ".join(
            seg.text.strip() for seg in segments if seg.text.strip()
        )
        return transcript if transcript else "(No speech detected)"
    except Exception as e:
        return f"Error during transcription: {str(e)}"

def generate_image_fn(prompt):
    if not prompt or prompt.startswith("❌"):
        return None
    try:
        image = image_pipe(prompt).images[0]
        torch.cuda.empty_cache()
        return image
    except Exception as e:
        print(f"Image error: {e}")
        return None

def generate_video_fn(prompt):
    if not prompt or prompt.startswith("❌"):
        return None
    try:
        output = video_pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_frames=16,
            guidance_scale=2.0,
            num_inference_steps=6,
            generator=torch.Generator(device="cpu").manual_seed(0),
        )
        gif_path = "/content/generated_video.gif"
        export_to_gif(output.frames[0], gif_path)
        torch.cuda.empty_cache()
        return gif_path
    except Exception as e:
        print(f"Video error: {e}")
        return None

def process_all(audio_file):
    text = transcribe_voice(audio_file)
    if text.startswith("❌") or text == "(No speech detected)":
        return text, None, None
    img = generate_image_fn(text)
    vid = generate_video_fn(text)
    return text, img, vid

print("\n[4/4] Launching Gradio Web App...")
with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), title="Voice2Vision AI") as demo:
    gr.Markdown("# 🎙️ Voice2Vision AI\n**Speak a scene — watch it come alive as an Image & Video.**")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Speak or Upload Audio")
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="Record or Upload Audio"
            )
            
            btn_transcribe = gr.Button("📝 1. Transcribe Voice", variant="secondary")
            prompt_box = gr.Textbox(label="Transcribed Prompt (editable)", lines=2)
            
            with gr.Row():
                btn_image = gr.Button("🖼️ Generate Image", variant="primary")
                btn_video = gr.Button("🎬 Generate Video", variant="primary")
            
            btn_all = gr.Button("✨ Do All at Once (Transcribe + Image + Video)", variant="stop")
            
        with gr.Column(scale=1):
            gr.Markdown("### 2. Generated Outputs")
            img_output = gr.Image(label="Generated Image (Dreamlike Diffusion, No Watermark)")
            vid_output = gr.Image(label="Generated Video (AnimateDiff 16-Frame GIF)")

    # Wiring Event Listeners
    btn_transcribe.click(fn=transcribe_voice, inputs=audio_input, outputs=prompt_box)
    btn_image.click(fn=generate_image_fn, inputs=prompt_box, outputs=img_output)
    btn_video.click(fn=generate_video_fn, inputs=prompt_box, outputs=vid_output)
    btn_all.click(fn=process_all, inputs=audio_input, outputs=[prompt_box, img_output, vid_output])

demo.launch(share=True, debug=True)